In [2]:
import os
import re

import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
import sys 
import analysis_utils
sys.path.append("../")

from checkpoint import CheckPoint
from datasets import LetterStringDataLoader
import generate_data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def bootstrapped_confint(arr, n_samples:int = 10_000, alpha:float=0.05):
    # sample with replacement n_samples times and take the mean of each sample:
    samples = np.array([np.mean(np.random.choice(arr, size=len(arr), replace=True)) for _ in range(n_samples)])
    # sort means in ascending order:
    samples.sort()
    # get bootstrapped confidence interval boundaries:
    confint_low = samples[int(n_samples * (alpha / 2))]
    confint_upp = samples[int(n_samples * (1 - alpha / 2))]
    return confint_low, confint_upp

In [4]:
# batching experiments no copy tasks, 20 permuted training alphabets:

# batching experiments with copy tasks, 20 permuted training alphabets:

# batching experiments with copy tasks, 200 permuted training alphabets:
pattern = r'perm(\d+)_'
dir_path = "../models/num_permuted_alphabets"
all_cp_paths = os.listdir(dir_path)
cps_copy_perm200 = []

for filename in all_cp_paths:
    if "200" in filename:
        cp = CheckPoint.from_pt("/".join([dir_path,filename]))
        cp.num_perm_alphs = 200
        cps_copy_perm200.append(cp)

In [11]:
for cp in cps_copy_perm200:
    print(cp.train_config.filename_model)

MLC_batchalph_dallstudy1_copy_perm200_nep20.pt
MLC_batchalph_dallstudy1_copy_perm200_nep20_rep1.pt
MLC_batchalph_dallstudy1_copy_perm200_nep20_rep2.pt
MLC_batchalph_dallstudy1_copy_perm200_nep20_rep3.pt
MLC_batchalph_dallstudy1_copy_perm200_nep20_rep4.pt
MLC_batchrand_dallstudy1_copy_perm200_nep20.pt
MLC_batchrand_dallstudy1_copy_perm200_nep20_rep1.pt
MLC_batchrand_dallstudy1_copy_perm200_nep20_rep2.pt
MLC_batchrand_dallstudy1_copy_perm200_nep20_rep3.pt
MLC_batchrand_dallstudy1_copy_perm200_nep20_rep4.pt


In [ ]:
def get_accuracy_table(checkpoints:list):
    # set up dataset to store accuracies:
    all_accs = pd.DataFrame(
        columns=["filename_model", "batching_method", "num. seen alphabets in training", "seen transform.", "new transform.", "alphabets"]
    )
    # new alphabets are constant across all checkpoints
    test_new_alph = LetterStringDataLoader(
        mode="test", 
        data_dir="../data/all_transformations_study1_new_alphabets",
        batch_size=2000
    )
    for cp in checkpoints:
        model = cp.load_model(verbose=False)

        # obtain accuracies on seen training alphabets:
        test_seen_alph = LetterStringDataLoader(
            mode="test", 
            data_dir=f"../{cp.train_config.dir_data}",
            batch_size=2000
        )
        pred = analysis_utils.predict_dataset(test_seen_alph, model, alternative_rule_errors=False)
        # exclude standard alphabet:
        pred = pred[pred["n_perm"]!= 0]
        test_acc_seen_transform = np.mean(pred[pred.distribution == "in"]["correct"])
        test_acc_new_transform = np.mean(pred[pred.distribution == "out-of"]["correct"])
        all_accs.loc[all_accs.shape[0],:] = [
            cp.train_config.filename_model, 
            cp.train_config.batching_method, 
            cp.num_perm_alphs, 
            test_acc_seen_transform, 
            test_acc_new_transform,
            "seen"
        ]

        # obtain accuracies on new alphabets:
        pred = analysis_utils.predict_dataset(test_new_alph, model, alternative_rule_errors=False)
        # exclude standard alphabet:
        pred = pred[pred["n_perm"]!= 0]
        test_acc_seen_transform = np.mean(pred[pred.distribution == "in"]["correct"])
        test_acc_new_transform = np.mean(pred[pred.distribution == "out-of"]["correct"])
        all_accs.loc[all_accs.shape[0],:] = [
            cp.train_config.filename_model, 
            cp.train_config.batching_method, 
            cp.num_perm_alphs, 
            test_acc_seen_transform, 
            test_acc_new_transform,
            "new"
        ]
        print(all_accs)
    return all_accs

In [15]:
tbl_copy_perm200 = get_accuracy_table(cps_copy_perm200)

Generating predictions:   0%|          | 0/22 [00:00<?, ?it/s]c:\Users\13579681\AppData\Local\anaconda3\envs\mlc-ls\Lib\site-packages\torch\nn\modules\transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(
Generating predictions: 23it [10:05, 26.32s/it]                        
Generating predictions: 6it [01:34, 15.73s/it]                       


                                   filename_model batching_method  \
0  MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   
1  MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   

  num. seen alphabets in training seen transform. new transform. alphabets  
0                             200         0.99992       0.361119      seen  
1                             200        0.998728       0.262257       new  


Generating predictions: 23it [09:02, 23.58s/it]                        
Generating predictions: 6it [01:47, 17.89s/it]                       


                                      filename_model batching_method  \
0     MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   
1     MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   
2  MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
3  MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   

  num. seen alphabets in training seen transform. new transform. alphabets  
0                             200         0.99992       0.361119      seen  
1                             200        0.998728       0.262257       new  
2                             200        0.893468        0.31361      seen  
3                             200        0.679281       0.036531       new  


Generating predictions: 23it [10:35, 27.65s/it]                        
Generating predictions: 6it [02:29, 24.95s/it]                       


...

                                       filename_model batching_method  \
0      MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   
1      MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   
2   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
3   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
4   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
5   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
6   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
7   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
8   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
9   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
10     MLC_batchrand_dallstudy1_copy_perm200_nep20.pt          random   
11     MLC_batchrand_dallstudy1_copy_perm200_nep20.pt          random   
12  MLC_batchrand_dallstudy1_copy_perm200_nep20_re.

Generating predictions: 23it [13:59, 36.51s/it]                        
Generating predictions: 6it [02:33, 25.62s/it]                       


                                       filename_model batching_method  \
0      MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   
1      MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   
2   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
3   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
4   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
5   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
6   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
7   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
8   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
9   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
10     MLC_batchrand_dallstudy1_copy_perm200_nep20.pt          random   
11     MLC_batchrand_dallstudy1_copy_perm200_nep20.pt          random   
12  MLC_batchrand_dallstudy1_copy_perm200_nep20_re.

Generating predictions: 23it [11:25, 29.81s/it]                        
Generating predictions: 6it [02:43, 27.23s/it]                       


                                       filename_model batching_method  \
0      MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   
1      MLC_batchalph_dallstudy1_copy_perm200_nep20.pt        alphabet   
2   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
3   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
4   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
5   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
6   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
7   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
8   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
9   MLC_batchalph_dallstudy1_copy_perm200_nep20_re...        alphabet   
10     MLC_batchrand_dallstudy1_copy_perm200_nep20.pt          random   
11     MLC_batchrand_dallstudy1_copy_perm200_nep20.pt          random   
12  MLC_batchrand_dallstudy1_copy_perm200_nep20_re.

In [16]:
tbl_copy_perm200

,filename_model,batching_method,num. seen alphabets in training,seen transform.,new transform.,alphabets
0,MLC_batchalph_dallstudy1_copy_perm200_nep20.pt,alphabet,200,0.99992,0.361119,seen
1,MLC_batchalph_dallstudy1_copy_perm200_nep20.pt,alphabet,200,0.998728,0.262257,new
2,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.893468,0.31361,seen
3,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.679281,0.036531,new
4,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,1.0,0.327959,seen
5,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.999682,0.194963,new
6,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,1.0,0.286832,seen
7,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.993958,0.098058,new
8,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.999761,0.326567,seen
9,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.99841,0.147664,new


In [17]:
tbl_copy_perm200.to_csv("copy_perm200_accuracies.csv", index=False)

In [18]:
pd.read_csv("copy_perm200_accuracies.csv")

,filename_model,batching_method,num. seen alphabets in training,seen transform.,new transform.,alphabets
0,MLC_batchalph_dallstudy1_copy_perm200_nep20.pt,alphabet,200,0.999920,0.361119,seen
1,MLC_batchalph_dallstudy1_copy_perm200_nep20.pt,alphabet,200,0.998728,0.262257,new
2,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.893468,0.313610,seen
3,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.679281,0.036531,new
4,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,1.000000,0.327959,seen
5,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.999682,0.194963,new
6,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,1.000000,0.286832,seen
7,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.993958,0.098058,new
8,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.999761,0.326567,seen
9,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.998410,0.147664,new
